# Masking Strategies

`MaskingStrategy` selects which positions to mask before feeding sequences to a masked language model sampler (ESM2, ESM3). It controls **which** positions get masked and **how many**.

## Methods

| Method | How it scores positions | Needs a model? |
|--------|------------------------|----------------|
| `"random"` (default) | Uniform — all positions equally likely | No |
| `"entropy"` | Shannon entropy from model logits — high uncertainty = more likely | Yes |
| `"max-logit"` | Negated max-logit — low confidence = more likely | Yes |

## How many positions to mask (pick one)

| Parameter | Behavior |
|-----------|----------|
| `num_mutations=N` | Exactly N positions |
| `mask_fraction=0.15` | ~15% of designable positions |
| *(neither)* | Default: ~30% of designable positions |

In [1]:
from bio_programming_tools.tools.masked_models.masking import MaskingStrategy

## 1. Random masking (default)

Mask ~30% of positions uniformly at random.

In [2]:
sequences = ["MVLSPADKTNVKAAWGKVGA"]

strategy = MaskingStrategy()
masked = strategy.mask(sequences)

print(f"Original: {sequences[0]}")
print(f"Masked:   {masked[0]}")
print(f"Positions masked: {sum(c == '_' for c in masked[0])}/{len(sequences[0])}")

Original: MVLSPADKTNVKAAWGKVGA
Masked:   M_LSPA_K_NV_AAW_K_GA
Positions masked: 6/20


## 2. Controlling mutation count

Use `num_mutations` for an exact count, or `mask_fraction` for a proportion.

In [3]:
# Exact count — reuse the same strategy object across calls
exact_strategy = MaskingStrategy(num_mutations=3)
print(f"num_mutations=3: {exact_strategy.mask(sequences)[0]}")
print(f"  (again):       {exact_strategy.mask(sequences)[0]}")

# Fraction
frac_strategy = MaskingStrategy(mask_fraction=0.5)
masked = frac_strategy.mask(sequences)
print(f"mask_fraction=0.5: {masked[0]}  ({sum(c == '_' for c in masked[0])} masked)")

num_mutations=3: _VLSPAD_TNVKAA_GKVGA
  (again):       MVLSP__KTNVKAAW_KVGA
mask_fraction=0.5: M_LSPADK__V_A_____G_  (10 masked)


## 3. Protecting positions with `fixed_positions`

Use 1-indexed `fixed_positions` to keep specific residues unchanged. This is useful for preserving catalytic sites, binding residues, or other functionally important positions.

In [4]:
# Fix positions 1, 2, 3 (M, V, L) — they will never be masked
strategy = MaskingStrategy(mask_fraction=1.0, fixed_positions=[1, 2, 3])
masked = strategy.mask(sequences)

print(f"Original:  {sequences[0]}")
print(f"Masked:    {masked[0]}")
print(f"Positions 1-3 preserved: {masked[0][:3]}")

Original:  MVLSPADKTNVKAAWGKVGA
Masked:    MVL_________________
Positions 1-3 preserved: MVL


Note that `mask_fraction` refers to **designable** positions (those not fixed), not the total sequence length.

In [5]:
seq = "MVLSPADKTNVKAAWGKVGA"  # 20 residues

# Fix 10 positions → 10 designable remain
# mask_fraction=0.5 → mask 50% of 10 designable = 5 positions
strategy = MaskingStrategy(
    mask_fraction=0.5,
    fixed_positions=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
)
masked = strategy.mask([seq])

n_masked = sum(c == "_" for c in masked[0])
print(f"Original:          {seq}")
print(f"Masked:            {masked[0]}")
print(f"Fixed (1-10):      {masked[0][:10]}")
print(f"Designable:        10 positions (11-20)")
print(f"Masked:            {n_masked} = 50% of 10 designable")

Original:          MVLSPADKTNVKAAWGKVGA
Masked:            MVLSPADKTNV_A___K_GA
Fixed (1-10):      MVLSPADKTN
Designable:        10 positions (11-20)
Masked:            5 = 50% of 10 designable


## 4. Batch masking

`mask()` accepts a list of sequences and masks each independently.

In [6]:
batch = [
    "MVLSPADKTNVKAAWGKVGA",
    "MNIFEMLRIDEGLRLKIYKD",
    "MKWVTFISLLFLFSSAYSRG",
]

strategy = MaskingStrategy(num_mutations=5)
masked_batch = strategy.mask(batch)

for orig, masked in zip(batch, masked_batch):
    diffs = [i + 1 for i, (a, b) in enumerate(zip(orig, masked)) if a != b]
    print(f"{orig} -> {masked}  (masked at positions {diffs})")

MVLSPADKTNVKAAWGKVGA -> MVLSP_DKTN_K__WGKV_A  (masked at positions [6, 11, 13, 14, 19])
MNIFEMLRIDEGLRLKIYKD -> M_IF__LRIDEGLRLKI_K_  (masked at positions [2, 5, 6, 18, 20])
MKWVTFISLLFLFSSAYSRG -> _KWVTFISL_F_FSSA_SR_  (masked at positions [1, 10, 12, 17, 20])


## 5. Model-guided masking (entropy / max-logit)

The `"entropy"` and `"max-logit"` methods use a protein language model's logits to score positions. Higher entropy or lower confidence positions are preferentially masked.

These methods require a GPU and a model. When used standalone (outside a sampling tool), set `model_name` explicitly.

The `temperature` parameter controls how strongly scores influence selection:
- `< 1.0`: greedy (strongly prefer highest-scored positions)
- `= 1.0`: use scores as-is (default)
- `> 1.0`: more uniform (diminish score differences)

In [7]:
# Entropy-based: mask the 5 most uncertain positions (requires GPU + model)
entropy_strategy = MaskingStrategy(
    method="entropy",
    model_name="esm2",
    num_mutations=5,
)
masked = entropy_strategy.mask(["MVLSPADKTNVKAAWGKVGA"])
print(f"Entropy-masked: {masked[0]}")

# Max-logit: mask the 5 least confident positions
maxlogit_strategy = MaskingStrategy(
    method="max-logit",
    model_name="esm2",
    num_mutations=5,
)
masked = maxlogit_strategy.mask(["MVLSPADKTNVKAAWGKVGA"])
print(f"MaxLogit-masked: {masked[0]}")

# Greedy entropy: temperature < 1 makes selection more deterministic
greedy_strategy = MaskingStrategy(
    method="entropy",
    model_name="esm2",
    num_mutations=5,
    temperature=0.1,
)
masked = greedy_strategy.mask(["MVLSPADKTNVKAAWGKVGA"])
print(f"Greedy entropy:  {masked[0]}")

Entropy-masked: M_LS_ADKT_VK_A_GKVGA


MaxLogit-masked: MVLS_ADKT_VK_A_GKV_A


Greedy entropy:  MVLSPAD_T_VKAA_G_V_A


## 6. Using with sampling tools

In practice, `MaskingStrategy` is passed as a config field to sampling tools like `run_esm2_sample`. The tool's `preprocess()` hook handles model setup automatically — you don't need to set `model_name`.

In [8]:
from bio_programming_tools import ESM2SampleConfig, ESM2SampleInput, run_esm2_sample

# The sampling tool applies the masking strategy internally
config = ESM2SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
)

result = run_esm2_sample(
    ESM2SampleInput(sequences=["MVLSPADKTNVKAAWGKVGA"]),
    config,
)
print(f"Sampled: {result.sequences[0]}")

Sampled: MKLSPADKTVVKAATGKMTA


## 7. Pre-masked sequence input

You can also skip `MaskingStrategy` entirely and pass pre-masked sequences directly to a sampling tool. Use `_` to mark positions for re-design. The tool detects the existing mask tokens and samples at those positions without applying any masking strategy.

In [9]:
# Manually mask positions 5-8 with underscores
pre_masked = "MVLS____TNVKAAWGKVGA"

result = run_esm2_sample(
    ESM2SampleInput(sequences=[pre_masked]),
    ESM2SampleConfig(),  # no masking_strategy needed
)
print(f"Input:   {pre_masked}")
print(f"Sampled: {result.sequences[0]}")
print(f"Only positions 5-8 were re-designed")

Input:   MVLS____TNVKAAWGKVGA
Sampled: MVLSEIQFTNVKAAWGKVGA
Only positions 5-8 were re-designed
